In [0]:
df = spark.table("bankingdata.silver.transaction")
display(df.limit(10))

### 1. Daily Transaction Summary

Business Use:
- Total transactions per day
- Total revenue
- High-value monitoring

In [0]:
from pyspark.sql.functions import *

df_gold_daily = (
    df
    .groupBy("event_date")
    .agg(
        count("*").alias("total_transactions"),
        sum("amount").alias("total_amount"),
        avg("amount").alias("avg_amount"),
        sum(when(col("is_high_value") == True, 1).otherwise(0)).alias("high_value_txn_count")
    )
    .withColumn("id", col("event_date").cast("string"))
    .withColumn("pk", col("event_date").cast("string"))
)

### 2. Customer Spending Summary

Business Use:
- Customer behavior
- High-value customers

In [0]:
df_gold_customer = (
    df
    .groupBy("customer_id")
    .agg(
        count("*").alias("total_transactions"),
        sum("amount").alias("total_spent"),
        avg("amount").alias("avg_spent"),
        max("amount").alias("max_transaction")
    )
    .withColumn("id", col("customer_id").cast("string"))
    .withColumn("pk", col("customer_id").cast("string"))
)

### 3. Transaction Type Analysis

Business Use:
- Debit vs Credit analysis

In [0]:
df_gold_type = (
    df
    .groupBy("transaction_direction")
    .agg(
        count("*").alias("txn_count"),
        sum("amount").alias("total_amount")
    )
    .withColumn("id", col("transaction_direction"))
    .withColumn("pk", col("transaction_direction"))
)

### 4. Location-Based Analysis

Business Use:
- City-wise revenue
- Fraud patterns

In [0]:
df_gold_location = (
    df
    .groupBy("location")
    .agg(
        count("*").alias("txn_count"),
        sum("amount").alias("total_amount")
    )
    .withColumn("id", col("location"))
    .withColumn("pk", col("location"))
)

### 5. Fraud / Suspicious Transactions


Business Use:
- Risk monitoring

In [0]:
df_gold_fraud = (
    df
    .filter(col("is_high_value") == True)
    .groupBy("event_date")
    .agg(
        count("*").alias("high_value_txn"),
        sum("amount").alias("total_high_value_amount")
    )
    .withColumn("id", col("event_date").cast("string"))
    .withColumn("pk", col("event_date").cast("string"))
)

### Save Gold Tables

In [0]:
df_gold_daily.write.format("delta").mode("append").saveAsTable("bankingdata.gold.daily_summary")

df_gold_customer.write.format("delta").mode("append").saveAsTable("bankingdata.gold.customer_summary")

df_gold_type.write.format("delta").mode("append").saveAsTable("bankingdata.gold.transaction_type_summary")

df_gold_location.write.format("delta").mode("append").saveAsTable("bankingdata.gold.location_summary")

df_gold_fraud.write.format("delta").mode("append").saveAsTable("bankingdata.gold.fraud_summary")